# 10年定着予測 - 3因子リスク要因数（L2×M×残業時間、58_）

## 位置づけ: `57_`の重要度分析で発見した第3因子を追加検証

`57_`（全特徴量キッチンシンク）で重要度上位に残業時間の各種統計が並んでいたことを受け、
`54_`で確認済みのL2×Mリスク要因数（0/1/2）に、**残業時間（後期12-23月・上位25%）**を
第3因子として追加した場合の効果をローカルEDAで確認した。

## ローカルEDAでの生の効果量（採否の根拠にはしない — [[local-test-not-decisive-for-ablation]]）

| リスク要因数(3因子) | n | 定着率 |
|---|---|---|
| 0個 | 1555 | 73.5% |
| 1個 | 960 | 37.4% |
| 2個 | 231 | 23.8% |
| 3個 | 15 | 13.3% |

**Cochran-Armitage傾向検定: z=-20.24, p≈0（機械精度の限界）。**
既存のL2×Mリスク要因数（0/1/2）との相関はわずか0.058で、独立した情報。

## 正直なリスク評価（L2・Mより弱い理論的根拠）

L2・Mが効いたのは「単体では木が学習しにくい比較型の情報、あるいはモデルに存在しなかった情報」
だったから。**残業時間は既に生の数値特徴量（`残業時間_late_mean`等）としてモデルに入っており、
単純な閾値分割は決定木が最も得意とする操作**なので、L2xMほどの新規性は期待できない。
また残業時間関連の特徴量は`52_`のdip-recovery特徴量が既にPublicで不採用になった前歴がある
（ただし52_は「形状」、今回は「後期の高さ」で定式化が異なる）。

## 追加した特徴量（2列、`OT3`グループ）

- `残業時間_後期高フラグ`：後期(12-23月)平均が上位25%かどうかの二値フラグ
  （閾値はTrainのみでfit）
- `L2xM_リスク要因数_3因子`：既存の`L2xM_リスク要因数`(0/1/2) + 上記フラグ = 0〜3の順序尺度

## 実行構成

`54_`の2構成それぞれに、第3因子追加ありなしの2パターンを回す（計4構成）。

## 判定

- 採否はPublicのみ
- 比較対象は`54_`の実測Public（441+LM列: 0.515030 / 113+LM列: 0.518492）
- 単一の事前登録済み検証として扱う。閾値をチューニングしない

> ⚠️ **ローカルMacで先行実行しないこと。**


In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 14.6 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "58_risk3_overtime"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-15 09:14:38] [INFO] === [58_risk3_overtime] 実験開始 ===


INFO:58_risk3_overtime:=== [58_risk3_overtime] 実験開始 ===


[2026-08-15 09:14:40] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


INFO:58_risk3_overtime:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


[2026-08-15 09:14:40] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/58_risk3_overtime_checkpoint.csv


INFO:58_risk3_overtime:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/58_risk3_overtime_checkpoint.csv


[2026-08-15 09:14:40] [INFO] チェックポイントは未作成（新規実行）


INFO:58_risk3_overtime:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-15 09:14:43] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:58_risk3_overtime:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-15 09:14:43] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:58_risk3_overtime:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-15 09:14:43] [INFO] 定着率: 0.5647


INFO:58_risk3_overtime:定着率: 0.5647


[2026-08-15 09:14:43] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:58_risk3_overtime:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-15 09:14:43] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:58_risk3_overtime:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-15 09:14:43] [INFO] Test  早期退職者: 0名 / 2502名


INFO:58_risk3_overtime:Test  早期退職者: 0名 / 2502名


[2026-08-15 09:14:43] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:58_risk3_overtime:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-15 09:14:43] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:58_risk3_overtime:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-15 09:14:43] [INFO] ------------------------------------------------------------


INFO:58_risk3_overtime:------------------------------------------------------------


[2026-08-15 09:14:43] [INFO] split非依存の基本特徴量を生成中...


INFO:58_risk3_overtime:split非依存の基本特徴量を生成中...


[2026-08-15 09:14:43] [INFO] ------------------------------------------------------------


INFO:58_risk3_overtime:------------------------------------------------------------


[2026-08-15 09:20:21] [INFO] split非依存の基本特徴量生成完了


INFO:58_risk3_overtime:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-15 09:20:21] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:58_risk3_overtime:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-15 09:20:22] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:58_risk3_overtime:入社時メモ: SVD累積寄与率=0.760


[2026-08-15 09:20:26] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:58_risk3_overtime:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-15 09:20:28] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:58_risk3_overtime:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-15 09:20:28] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:58_risk3_overtime:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-15 09:20:28] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:58_risk3_overtime:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-15 09:22:25] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:58_risk3_overtime:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-15 09:22:25] [INFO] Persona単位の基本特徴量を生成中...


INFO:58_risk3_overtime:Persona単位の基本特徴量を生成中...


[2026-08-15 09:22:25] [INFO] Persona単位の基本特徴量処理完了


INFO:58_risk3_overtime:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、49_でパーサーを修正）

`extract_workstyle_section()` に、見出し（`勤務地・働き方：`）が無い書式Bのフォールバックを追加した。
それ以外（`classify_reloc` / `extract_desired_location_v1` / `v2`）は `40_` と同一。


In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 49_: 見出しがない書式B（276件、5.24%）のフォールバック。
    # 「勤務地・転居・在宅勤務」に言及する行を拾い、疑似セクションとして返す。
    # 以降のclassify_reloc/extract_desired_location_v1/v2はre.searchで探すだけなので、
    # 複数行を連結してもそのまま動く。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


def _report_ws_coverage_fix(train_persona, test_persona):
    """49_の修正がどれだけカバー率を回復させたかをログに残す（診断専用、学習には影響しない）"""
    def old_fn(text):
        if pd.isna(text):
            return None
        m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
        return m.group(1).strip() if m else None

    all_persona = pd.concat([train_persona[["入社時メモ"]], test_persona[["入社時メモ"]]], ignore_index=True)
    ws_old = all_persona["入社時メモ"].apply(old_fn)
    ws_new = all_persona["入社時メモ"].apply(extract_workstyle_section)
    n = len(all_persona)
    logger.info(f"[49_診断] 見出し欠落 修正前 {ws_old.isna().sum()}件({ws_old.isna().sum()/n:.2%}) "
                f"→ 修正後 {ws_new.isna().sum()}件({ws_new.isna().sum()/n:.2%})")


_report_ws_coverage_fix(train_persona, test_persona)


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())


[2026-08-15 09:22:25] [INFO] [49_診断] 見出し欠落 修正前 276件(5.24%) → 修正後 2件(0.04%)


INFO:58_risk3_overtime:[49_診断] 見出し欠落 修正前 276件(5.24%) → 修正後 2件(0.04%)


[2026-08-15 09:22:25] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:58_risk3_overtime:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-15 09:22:25] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:58_risk3_overtime:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-15 09:22:25] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:58_risk3_overtime:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2397
1     364
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2385
1     376
Name: count, dtype: int64


## 5b. L2×M交互作用フラグ（`54_`で追加、Job Embeddedness理論由来）


In [14]:
# ============================================================
# 54_: L2×Mのリスク要因数（Job Embeddedness理論: 複数の埋め込み不足の重なり）
#   ブロックM（専攻職種の分析的ミスマッチ）は29_で単体却下済み（GBDT redundancy）。
#   L2（転居x勤務地_状態_v2 = "非許容_不一致"）との組み合わせをフラグ化する。
#
#   ローカルEDAでの生の効果量（Cochran-Armitage傾向検定, p≈0・機械精度限界）:
#     リスク要因0個(n=1885): 定着率67.0%
#     リスク要因1個(n=653) : 定着率28.8%
#     リスク要因2個(n=47)  : 定着率 2.1%
#   単なるAND(2個該当)だけでなく、0→1→2ときれいな段階的用量反応があったため、
#   二値フラグに加えて順序尺度の"risk_count"も特徴量として渡す。
#   ただし該当セルのp値は採用基準(p<1e-20)を厳密には満たさないセルもあり、
#   単一の事前登録済み検証として扱う（閾値をチューニングしない）。
# ============================================================

_ANALYTICAL_MAJOR = {"情報", "理工学"}
_ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_l2_m_interaction_features(persona_df, reloc_v2_df):
    is_analytical_major = persona_df["専攻分野"].isin(_ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(_ANALYTICAL_JOB)
    m_bad = (~is_analytical_major & is_analytical_job).astype(int)

    state = reloc_v2_df.set_index("社員ID").loc[persona_df["社員ID"], "転居x勤務地_状態_v2"].values
    l2_bad = (state == "非許容_不一致").astype(int)

    both_bad = (l2_bad & m_bad)
    risk_count = l2_bad + m_bad  # 0/1/2の順序尺度（用量反応をそのまま渡す）

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "M_不適合": m_bad,
        "L2xM_ダブル不適合": both_bad,
        "L2xM_リスク要因数": risk_count,
    })


train_l2m = create_l2_m_interaction_features(train_persona, train_reloc_v2)
test_l2m = create_l2_m_interaction_features(test_persona, test_reloc_v2)
logger.info(f"L2xMリスク特徴量: Train {train_l2m.shape}, Test {test_l2m.shape}")
print(train_l2m["L2xM_リスク要因数"].value_counts().sort_index())


[2026-08-15 09:22:25] [INFO] L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


INFO:58_risk3_overtime:L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


L2xM_リスク要因数
0    2025
1     689
2      47
Name: count, dtype: int64


## 5c. 3因子リスク要因数（`58_`で追加、残業時間を第3因子に）


In [15]:
# ============================================================
# 58_: 残業時間を第3因子に加えたリスク要因数（0〜3）
#   57_の重要度分析で発見。既存のL2xM_リスク要因数との相関は0.058と独立。
#   Cochran-Armitage傾向検定: z=-20.24, p≈0（ローカルEDA、採否の根拠にはしない）。
#   閾値(75%点)はTrainのみでfitする（リーク防止）。
# ============================================================

_overtime_late_threshold = train_monthly_agg["残業時間_late_mean"].quantile(0.75)
logger.info(f"残業時間_late_mean の75%点(Trainでfit): {_overtime_late_threshold:.2f}")


def create_risk3_features(persona_df, monthly_agg_df, l2m_df, threshold):
    late = monthly_agg_df.set_index("社員ID").loc[persona_df["社員ID"], "残業時間_late_mean"]
    overtime_flag = (late.values >= threshold).astype(int)
    risk2 = l2m_df.set_index("社員ID").loc[persona_df["社員ID"], "L2xM_リスク要因数"].values
    risk3 = risk2 + overtime_flag
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "残業時間_後期高フラグ": overtime_flag,
        "L2xM_リスク要因数_3因子": risk3,
    })


train_risk3 = create_risk3_features(train_persona, train_monthly_agg, train_l2m, _overtime_late_threshold)
test_risk3 = create_risk3_features(test_persona, test_monthly_agg, test_l2m, _overtime_late_threshold)
logger.info(f"3因子リスク要因数: Train {train_risk3.shape}, Test {test_risk3.shape}")
print(train_risk3["L2xM_リスク要因数_3因子"].value_counts().sort_index())


[2026-08-15 09:22:25] [INFO] 残業時間_late_mean の75%点(Trainでfit): 17.79


INFO:58_risk3_overtime:残業時間_late_mean の75%点(Trainでfit): 17.79


[2026-08-15 09:22:25] [INFO] 3因子リスク要因数: Train (2761, 3), Test (2502, 3)


INFO:58_risk3_overtime:3因子リスク要因数: Train (2761, 3), Test (2502, 3)


L2xM_リスク要因数_3因子
0    1548
1     963
2     237
3      13
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [16]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        tf = tf.merge(train_l2m, on=ID_COL, how="left")
        tf = tf.merge(train_risk3, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_l2m, on=ID_COL, how="left")
        ttf = ttf.merge(test_risk3, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


## 7. チェックポイント機能（`18_`〜`28_`をベースに、37_で固定スキーマ化）

`28_`までは全configが同じキーを持っていたが、37_ は A / BC / D で記録すべき情報が異なる。
キー構成がバラバラのまま `mode="a"` でCSVに追記すると列がずれて壊れるため、
`RESULT_SCHEMA` に揃えてから書き出す。

In [17]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


## 8. モデル関数（37_版）

`28_`の `run_model_config` を3つに分解する。

- `tune_hyperparams`: Optunaで探索（探索空間は`18_`〜`28_`と完全に同一、n_trials=25）
- `fit_holdout`: 80/20で学習し、early stoppingで最良反復数を決める。シードを変えて複数回実行できる
- `fit_full_train`: **Train全件**で学習する（検証セットが無いので反復数は固定、early stoppingなし）

シード平均は、同一パラメータ・同一特徴量のままシードだけ変えたモデルの**予測確率を単純平均**する。
重みを一切学習しないので、`11_`/`12_`/`32_`で失敗した「OOFから重みを学習するアンサンブル」とは
別物であり、過去の教訓には抵触しない。

In [18]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 9. 特徴量の組み立て

ブロックは `L2`（= `28_`の `L_v2_extended`、現在の最良）に固定する。

- `split_80_20` × 検証=全体 → config A（`28_`の完全再現）
- `split_80_20` × 検証=生存者のみ → config B / C
- `split_100`（全件） → config D / D2

In [19]:
BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[A用] split_80_20 / 検証=全体（28_と同一）")
ag_train_80, ag_val_all, test_features = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=False)

logger.info("[B,C用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[D用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"A: train={len(ag_train_80)}, val={len(ag_val_all)}（早期退職者を含む）")
logger.info(f"B/C: train={len(ag_train_80b)}, val={len(ag_val_surv)}（生存者のみ）")
logger.info(f"D: train={len(ag_full)}（全件）, val={len(ag_empty)}（空）")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80))}")

# 生存者マスク（Aの検証予測を生存者だけで採点し直すのに使う）
SURV_MASK_A = ~ag_val_all.index.isin(EARLY_LEAVER_IDS)
assert len(ag_train_80) == len(ag_train_80b), "A と B/C の学習データは同一のはず"
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"

[2026-08-15 09:22:25] [INFO] ============================================================


INFO:58_risk3_overtime:============================================================


[2026-08-15 09:22:25] [INFO] [A用] split_80_20 / 検証=全体（28_と同一）


INFO:58_risk3_overtime:[A用] split_80_20 / 検証=全体（28_と同一）


[2026-08-15 09:22:25] [INFO] [B,C用] split_80_20 / 検証=生存者のみ


...

INFO:58_risk3_overtime:B/C: train=2208, val=535（生存者のみ）


[2026-08-15 09:22:26] [INFO] D: train=2761（全件）, val=0（空）


INFO:58_risk3_overtime:D: train=2761（全件）, val=0（空）


[2026-08-15 09:22:26] [INFO] 特徴量数: 446


INFO:58_risk3_overtime:特徴量数: 446


## 10. 特徴量グループの棚卸し

In [20]:
# ============================================================
# 特徴量グループの棚卸し
#   prepare_split() が merge している元フレームごとに列を分類する。
#   「どのグループにも属さない列」「2グループに重複する列」が出たら
#   減量の定義がずれているのでassertで止める。
# ============================================================

ALL_FEATS = set(_feature_cols(ag_train_80))

# prepare_split() 内で生成される派生列（元フレームを持たないのでここに明示）
DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
# 部署Target Encoding が生む列（create_department_target_encoding の出力）
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "OT3":       _cols_of(train_risk3),
    "LM":        _cols_of(train_l2m),
    "derived":   DERIVED_COLS,
}

# 実際に特徴量として残っている列だけに絞る（drop_colsで消えたものを自動的に除外）
FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 446 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  LM            3 列   例: ['M_不適合', 'L2xM_ダブル不適合']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  OT3           2 列   例: ['残業時間_後期高フラグ', 'L2xM_リスク要因数_3因子']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
-----

## 11. 月次集約の「指標 × 統計」分解

In [21]:
# ============================================================
# 月次集約(agg)の「指標 × 統計」分解
#   create_monthly_aggregation_features が作る 16指標 × 14統計 を分解し、
#   冗長な統計を落とせるようにする。
# ============================================================

AGG_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
AGG_ALL_STATS = [
    "mean", "std", "min", "max", "median", "cv",
    "early_mean", "mid_mean", "late_mean", "late_minus_early", "late_early_ratio",
    "slope", "diff", "ratio",
]

# 残す統計。冗長性を根拠に選ぶ（検証スコアで選んでいない）:
#   median←mean と重複 / cv←std/mean の比 / min,max←外れ値1点 /
#   mid_mean←early,lateから内挿可能 / late_minus_early,late_early_ratio,diff,ratio←slopeと同義
AGG_KEEP_STATS = {"mean", "std", "early_mean", "late_mean", "slope"}


def _agg_stat(col):
    """agg列名を (指標, 統計) に分解して統計名を返す。最長一致で指標を特定する。"""
    best = None
    for m in AGG_METRICS:
        if col.startswith(m + "_") and (best is None or len(m) > len(best)):
            best = m
    if best is None:
        return None
    return col[len(best) + 1:]


_unmapped = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) not in AGG_ALL_STATS]
assert not _unmapped, f"指標×統計に分解できないagg列: {_unmapped}"

AGG_SLIM_COLS = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in AGG_KEEP_STATS]
print(f"agg: {len(FEATURE_GROUPS['agg'])} 列 → 統計を{sorted(AGG_KEEP_STATS)}に限定すると {len(AGG_SLIM_COLS)} 列")
print(f"落とす統計: {sorted(set(AGG_ALL_STATS) - AGG_KEEP_STATS)}")


agg: 224 列 → 統計を['early_mean', 'late_mean', 'mean', 'slope', 'std']に限定すると 80 列
落とす統計: ['cv', 'diff', 'late_early_ratio', 'late_minus_early', 'max', 'median', 'mid_mean', 'min', 'ratio']


## 12. 構成の事前登録（58_: OT3追加効果の測定に限定）

`54_`のCONFIGSから、既にPublicで判明済みの2構成（LM込み）を基準にし、OT3（3因子化）を追加した場合と比較する。


In [22]:
# ============================================================
# 構成の事前登録（実行前に確定させる。結果を見てから増減しない）
# ============================================================

A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}

ITER_HOLDOUT = 560   # 38_ で 80%学習(2208件)での最適点と実測。350〜900は平坦
ITER_FULL    = 560   # D3(Public 0.522659)と同一。ここを変えると「特徴量だけの差」でなくなる

SEEDS_SUB = [42, 2024, 7, 1234, 99]                      # 提出用。D3と同一の5シード
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]    # 検証用。8シードで分解能を稼ぐ
SEEDS_ES  = [42, 2024, 7]                                # early stopping診断用（遅いので3シード）

CORE_GROUPS = {"persona", "agg", "deptte", "derived", "L2"}

CONFIGS = {
    "R0_memofix":              {"groups": ALL_GROUPS - {"OT3"},  "agg_stats": None},
    "R0_memofix_plus_OT3":     {"groups": ALL_GROUPS,            "agg_stats": None},
    "R6_lean_memofix":         {"groups": CORE_GROUPS,           "agg_stats": AGG_KEEP_STATS},
    "R6_lean_memofix_plus_OT3":{"groups": CORE_GROUPS | {"OT3"}, "agg_stats": AGG_KEEP_STATS},
}

# 足切り基準（事前登録）: R0からこれ以上悪化した構成は提出しない
VAL_REJECT_MARGIN = 0.02


def cols_for(spec, df):
    """構成specに対応する特徴量列を、元データフレームの列順を保って返す。

    列順を保つのは R0_ref を D3 とビット単位で同じ入力にするため
    （CatBoostは特徴量の順序で分割候補の探索順が変わりうる）。
    """
    keep = set()
    for g in spec["groups"]:
        if g == "agg" and spec["agg_stats"] is not None:
            keep |= {c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in spec["agg_stats"]}
        else:
            keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


print(f"{'config':<14s} {'列数':>5s}  除外グループ")
print("-" * 72)
for name, spec in CONFIGS.items():
    dropped = sorted(ALL_GROUPS - spec["groups"])
    if spec["agg_stats"] is not None:
        dropped = dropped + ["agg統計を5種に限定"]
    print(f"{name:<14s} {len(cols_for(spec, ag_train_80)):>5d}  {', '.join(dropped) if dropped else '（なし）'}")


config            列数  除外グループ
------------------------------------------------------------------------
R0_memofix       444  OT3
R0_memofix_plus_OT3   446  （なし）
R6_lean_memofix   113  LM, OT3, advstats, catchange, cluster, domain, edafeat, mgr, missing, quarterly, tfidf, agg統計を5種に限定
R6_lean_memofix_plus_OT3   115  LM, advstats, catchange, cluster, domain, edafeat, mgr, missing, quarterly, tfidf, agg統計を5種に限定


## 13. モデル関数（反復数固定）

In [23]:
# ============================================================
# モデル関数（40_版: 反復数固定・特徴量列を明示的に受け取る）
# ============================================================

def _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed):
    model = cb.CatBoostClassifier(
        **params, iterations=int(n_iter), random_seed=seed,
        verbose=False, cat_features=obj_cols, task_type="CPU",
    )
    model.fit(X_tr, y_tr)
    return model


def fit_holdout_fixed(ag_train, ag_val, feature_cols, params, n_iter, seeds):
    """80/20ホールドアウトを反復数固定で学習し、シードごとの検証予測を返す。

    early stopping を使わないのは、38_ で best_iteration(448.6) が
    固定反復の真の最適点(560)を系統的に下回ると分かったため。
    """
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]

    val_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        val_preds.append(model.predict_proba(X_va)[:, 1])
    val_preds = np.array(val_preds)

    singles = [log_loss(y_va, vp) for vp in val_preds]
    return {
        "val_seedavg": float(log_loss(y_va, val_preds.mean(axis=0))),
        "val_single_mean": float(np.mean(singles)),
        "val_single_sd": float(np.std(singles)),
        "val_preds": val_preds,
        "y_val": y_va.values,
    }


def best_iter_diag(ag_train, ag_val, feature_cols, params, seeds):
    """診断専用: この特徴量セットでの early stopping 最適反復数。

    採否には使わない。ITER=560 が構成ごとに極端にズレていないかの安全確認。
    """
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]

    iters = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        iters.append(model.get_best_iteration())
    return float(np.mean(iters))


def fit_full_fixed(ag_full, test_feats, feature_cols, params, n_iter, seeds):
    """Train全件で学習して Test を予測（検証セットが無いので反復数固定）"""
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = ag_full[feature_cols].fillna(-999), ag_full[TARGET_COL]
    X_test = test_feats[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"    seed={seed}: 全件学習完了")
    return np.array(test_preds)


print("✅ モデル関数定義完了（fit_holdout_fixed / best_iter_diag / fit_full_fixed）")


✅ モデル関数定義完了（fit_holdout_fixed / best_iter_diag / fit_full_fixed）


In [24]:
# ============================================================
# チェックポイント（40_用にスキーマを差し替える）
#   make_row / load_checkpoint / save_checkpoint_row はグローバルの
#   RESULT_SCHEMA を参照するので、ここで上書きすれば流用できる。
# ============================================================

RESULT_SCHEMA = [
    "config", "kind", "n_features", "dropped_groups",
    "val_seedavg", "val_single_mean", "val_single_sd", "best_iter_es",
    "n_iterations", "n_train", "pred_mean", "submission_path",
]


def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label] if len(checkpoint) else checkpoint
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_seedavg={row.get('val_seedavg')}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result


# スキーマ差し替えが効いているかの往復テスト
_probe = make_row(config="__schema_probe__", kind="test", n_features=1)
assert list(_probe.keys()) == RESULT_SCHEMA, "make_rowが新スキーマを見ていない"

_rejected = False
try:
    make_row(config="x", val_score=0.5)   # 旧(37_)スキーマのキー。弾かれるはず
except AssertionError as _e:
    _rejected = "RESULT_SCHEMA" in str(_e)
assert _rejected, "旧スキーマのキーが素通りした。RESULT_SCHEMAの差し替えが効いていない"

print("✅ チェックポイントを40_スキーマに差し替え完了")
print(f"   {RESULT_SCHEMA}")


✅ チェックポイントを40_スキーマに差し替え完了
   ['config', 'kind', 'n_features', 'dropped_groups', 'val_seedavg', 'val_single_mean', 'val_single_sd', 'best_iter_es', 'n_iterations', 'n_train', 'pred_mean', 'submission_path']


## 14. 修正版特徴量での学習・検証

以降は`40_`の実行部分（モデル関数・チェックポイント・実行ループ）を無変更で流用する。


In [25]:
# ============================================================
# 減量構成の実行（R0 → R1 → R3 → R5 → R2 → R6）
#   全構成で共通:
#     - ハイパーパラメータ A_PARAMS 固定
#     - 検証   : 先頭80%学習 / 生存者535名 / 反復560固定 / 8シード平均
#     - 提出   : Train全件学習 / 反復560固定 / 5シード平均
#   つまり D3(Public 0.522659) との差は「特徴量」だけ。
# ============================================================

def make_reduction_runner(config_label, spec):
    def _run():
        feats = cols_for(spec, ag_train_80b)
        feats_full = cols_for(spec, ag_full)
        assert feats == feats_full, "80%学習と全件学習で特徴量列が食い違っている"
        dropped = sorted(ALL_GROUPS - spec["groups"])
        if spec["agg_stats"] is not None:
            dropped = dropped + ["agg_slim"]

        logger.info("=" * 60)
        logger.info(f"[{config_label}] {len(feats)}列 / 除外: {dropped or 'なし'}")

        bi = best_iter_diag(ag_train_80b, ag_val_surv, feats, A_PARAMS, SEEDS_ES)
        logger.info(f"  [診断] early stoppingの最適反復 ≈ {bi:.0f}（固定値{ITER_HOLDOUT}との比較用）")

        hold = fit_holdout_fixed(ag_train_80b, ag_val_surv, feats, A_PARAMS, ITER_HOLDOUT, SEEDS_VAL)
        logger.info(f"  検証(生存者{len(ag_val_surv)}名): シード平均 {hold['val_seedavg']:.6f} "
                    f"/ 単一シード {hold['val_single_mean']:.6f} ± {hold['val_single_sd']:.6f}")
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy", hold["val_preds"])

        test_preds = fit_full_fixed(ag_full, test_features_full, feats, A_PARAMS, ITER_FULL, SEEDS_SUB)
        preds = test_preds.mean(axis=0)
        path = save_submission(test_features_full.index, preds, config_label)

        return make_row(
            config=config_label, kind="reduction", n_features=len(feats),
            dropped_groups=",".join(dropped) if dropped else "",
            val_seedavg=hold["val_seedavg"], val_single_mean=hold["val_single_mean"],
            val_single_sd=hold["val_single_sd"], best_iter_es=bi,
            n_iterations=ITER_FULL, n_train=len(ag_full), pred_mean=float(preds.mean()),
            submission_path=path,
        )
    return _run


reduction_results = {}
for _name, _spec in CONFIGS.items():
    reduction_results[_name] = run_or_resume(_name, make_reduction_runner(_name, _spec))

print()
print(f"{'config':<14s} {'列数':>5s} {'val(8シード平均)':>16s} {'単一sd':>9s} {'ES最適反復':>10s}")
print("-" * 62)
for _name, _r in reduction_results.items():
    print(f"{_name:<14s} {int(_r['n_features']):>5d} {float(_r['val_seedavg']):>16.6f} "
          f"{float(_r['val_single_sd']):>9.6f} {float(_r['best_iter_es']):>10.0f}")


[2026-08-15 09:22:26] [INFO] ============================================================


INFO:58_risk3_overtime:============================================================


[2026-08-15 09:22:26] [INFO] [R0_memofix] 444列 / 除外: ['OT3']


INFO:58_risk3_overtime:[R0_memofix] 444列 / 除外: ['OT3']


[2026-08-15 09:22:39] [INFO]   [診断] early stoppingの最適反復 ≈ 417（固定値560との比較用）


...

[2026-08-15 09:26:17] [INFO]     seed=99: 全件学習完了


INFO:58_risk3_overtime:    seed=99: 全件学習完了


[2026-08-15 09:26:17] [INFO]   提出ファイル: 20260815_58_risk3_overtime_R6_lean_memofix_plus_OT3.csv（予測平均=0.5893）


INFO:58_risk3_overtime:  提出ファイル: 20260815_58_risk3_overtime_R6_lean_memofix_plus_OT3.csv（予測平均=0.5893）



config            列数      val(8シード平均)      単一sd     ES最適反復
--------------------------------------------------------------
R0_memofix       444         0.505477  0.005181        417
R0_memofix_plus_OT3   446         0.506052  0.005818        458
R6_lean_memofix   113         0.506397  0.006709        316
R6_lean_memofix_plus_OT3   115         0.508812  0.005992        295


In [26]:
# ============================================================
# サニティチェック: 修正前(D3/R0_ref, 441列)との予測差を見る
#   ここでは「一致すること」ではなく「小さいがゼロではない差」が出ることを確認する。
#   書式Bは全体の5.24%なので、変化するのはその近辺の行だけのはず。
#   相関が極端に低い/MADがノイズ床(0.02122, 41_実測)を超えるなら、
#   意図しない副作用が出ているのでバグを疑う。
# ============================================================

_r0_path = Path(reduction_results["R0_memofix"]["submission_path"])
_r0 = pd.read_csv(_r0_path, header=None, names=[ID_COL, "pred"])

_d3_candidates = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_37_full_train_seed_averaging_D3_Aparams_full_x125.csv"))

NOISE_FLOOR = 0.02122  # 41_実測: 同一構成の別実行間の95%上限

if _d3_candidates:
    _d3 = pd.read_csv(_d3_candidates[-1], header=None, names=[ID_COL, "pred"])
    _m = _r0.merge(_d3, on=ID_COL, suffixes=("_memofix", "_d3"))
    assert len(_m) == len(_r0), "社員IDが一致しない"
    _corr = _m["pred_memofix"].corr(_m["pred_d3"])
    _mad = (_m["pred_memofix"] - _m["pred_d3"]).abs().mean()
    print(f"D3(修正前)ファイル: {_d3_candidates[-1].name}")
    print(f"  相関           : {_corr:.6f}")
    print(f"  平均絶対差     : {_mad:.6f}（ノイズ床 {NOISE_FLOOR}）")
    print(f"  予測平均 修正後/D3 : {_m['pred_memofix'].mean():.4f} / {_m['pred_d3'].mean():.4f}")
    if _corr > 0.98 and 0 < _mad < NOISE_FLOOR:
        print("✅ 想定どおり: 小さいが非ゼロの差。書式B(5.24%)周辺の変化と整合的")
    elif _mad >= NOISE_FLOOR:
        print("⚠️ 差がノイズ床を超えている。意図しない副作用が無いか特徴量を確認すること")
    else:
        print("⚠️ 差がほぼゼロ。パーサー修正が反映されていない可能性がある")
else:
    print("⚠️ D3の提出ファイルが見つからなかった（サニティチェックをスキップ）")


D3(修正前)ファイル: 20260811_37_full_train_seed_averaging_D3_Aparams_full_x125.csv
  相関           : 0.978345
  平均絶対差     : 0.036444（ノイズ床 0.02122）
  予測平均 修正後/D3 : 0.5902 / 0.5875
⚠️ 差がノイズ床を超えている。意図しない副作用が無いか特徴量を確認すること


## 15. 結果まとめ

In [27]:
# ============================================================
# 結果まとめ: 3因子(OT3)追加の効果 と 54_実測との再現性確認
# ============================================================

HIST_54 = {
    "R0_memofix":      {"val": 0.505477, "public": 0.515030},
    "R6_lean_memofix": {"val": 0.508404, "public": 0.518492},
}

rows = []
for name, r in reduction_results.items():
    val = float(r["val_seedavg"])
    rows.append({
        "config": name,
        "列数": int(r["n_features"]),
        "val": val,
        "ファイル": Path(r["submission_path"]).name,
    })
summary = pd.DataFrame(rows).set_index("config")

print("=== 54_との再現性確認（base 2構成） ===")
for base in ["R0_memofix", "R6_lean_memofix"]:
    hist_val = HIST_54[base]["val"]
    now_val = summary.loc[base, "val"]
    print(f"{base:<28s} 54_実測val={hist_val:.6f}  今回val={now_val:.6f}  差={now_val-hist_val:+.6f}")

print()
print("=== OT3(3因子)追加の効果（同一セッション内の差分） ===")
pairs = [("R0_memofix", "R0_memofix_plus_OT3"), ("R6_lean_memofix", "R6_lean_memofix_plus_OT3")]
for base, plus in pairs:
    d = summary.loc[plus, "val"] - summary.loc[base, "val"]
    print(f"{plus:<28s} val差(対{base}) = {d:+.6f}")

summary["提出"] = "提出する"
for base, plus in pairs:
    d = summary.loc[plus, "val"] - summary.loc[base, "val"]
    if d > VAL_REJECT_MARGIN:
        summary.loc[plus, "提出"] = "見送り（足切り）"

pd.set_option("display.width", 200)
print()
print(summary.to_string())
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv")
logger.info(f"サマリを保存: {TODAY}_{SCRIPT_NAME}_summary.csv")

print()
print("=" * 70)
print("提出候補（_plus_OT3 のみが今回の新規検証対象。baseは再現性確認用で提出不要）")
print("=" * 70)
for name in ["R0_memofix_plus_OT3", "R6_lean_memofix_plus_OT3"]:
    if summary.loc[name, "提出"] == "提出する":
        base = {v: k for k, v in pairs}[name]
        _valdiff = summary.loc[name, "val"] - summary.loc[base, "val"]
        _hpub = HIST_54[base]["public"]
        print(f"{name:<28s} {summary.loc[name, 'ファイル']}")
        print(f"  対応するbase: {base} / val差 {_valdiff:+.6f} / 54_のPublic参照値: {_hpub:.6f}")


=== 54_との再現性確認（base 2構成） ===
R0_memofix                   54_実測val=0.505477  今回val=0.505477  差=+0.000000
R6_lean_memofix              54_実測val=0.508404  今回val=0.506397  差=-0.002007

=== OT3(3因子)追加の効果（同一セッション内の差分） ===
R0_memofix_plus_OT3          val差(対R0_memofix) = +0.000575
R6_lean_memofix_plus_OT3     val差(対R6_lean_memofix) = +0.002414

                           列数       val                                                     ファイル    提出
config                                                                                                
R0_memofix                444  0.505477                20260815_58_risk3_overtime_R0_memofix.csv  提出する
R0_memofix_plus_OT3       446  0.506052       20260815_58_risk3_overtime_R0_memofix_plus_OT3.csv  提出する
R6_lean_memofix           113  0.506397           20260815_58_risk3_overtime_R6_lean_memofix.csv  提出する
R6_lean_memofix_plus_OT3  115  0.508812  20260815_58_risk3_overtime_R6_lean_memofix_plus_OT3.csv  提出する
[2026-08-15 09:26:18] [INFO] サマリを保存: 2026

INFO:58_risk3_overtime:サマリを保存: 20260815_58_risk3_overtime_summary.csv



提出候補（_plus_OT3 のみが今回の新規検証対象。baseは再現性確認用で提出不要）
R0_memofix_plus_OT3          20260815_58_risk3_overtime_R0_memofix_plus_OT3.csv
  対応するbase: R0_memofix / val差 +0.000575 / 54_のPublic参照値: 0.515030
R6_lean_memofix_plus_OT3     20260815_58_risk3_overtime_R6_lean_memofix_plus_OT3.csv
  対応するbase: R6_lean_memofix / val差 +0.002414 / 54_のPublic参照値: 0.518492


## 16. 提出方針

### 判定（事前登録）

- **採否は Public のみ。** 検証は足切り（悪化検出）専用（[[validation-asymmetry]]）
- **足切り**: `VAL_REJECT_MARGIN`（0.02）を超えて悪化したら提出しない
- 2構成とも「OT3追加」という単一の介入なので、両方提出してよい

### 期待値について

残業時間は既に生特徴量としてモデルに入っており、L2・Mほどの新規性の理論的根拠は無い。ローカルEDAの傾向検定は非常に強い（p≈0）が、[[validation-asymmetry]]の教訓通り、検証の強さをそのまま信用しない。期待値は控えめに見る。

In [28]:
print([c for c in ag_full.columns if 'L2xM' in c or c == 'M_不適合'])
print(train_l2m.columns.tolist())
print(train_l2m.head())

['M_不適合', 'L2xM_ダブル不適合', 'L2xM_リスク要因数', 'L2xM_リスク要因数_3因子']
['社員ID', 'M_不適合', 'L2xM_ダブル不適合', 'L2xM_リスク要因数']
      社員ID  M_不適合  L2xM_ダブル不適合  L2xM_リスク要因数
0  E000001      0            0            0
1  E000005      0            0            0
2  E000007      0            0            1
3  E000008      0            0            0
4  E000010      0            0            1
